In [5]:
import pdfplumber
import pandas as pd
import re

# Input PDF
pdf_path = "SbiLong.pdf"

In [6]:
rows = []

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        tables = page.extract_tables()
        if not tables:
            continue
        
        for table in tables:
            for row in table:
                # Skip header rows / empty rows
                if not row or "Date" in str(row[0]):
                    continue
                
                rows.append(row)

# Create DataFrame
df = pd.DataFrame(rows, columns=["Date", "Details", "Ref No./Cheque No", "Debit", "Credit", "Balance"])

# Remove fully blank rows
df = df.dropna(how='all').reset_index(drop=True)

# ✅ Clean narration into one single line
df["Details"] = (
    df["Details"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)  # Merge line breaks & spaces
    .str.strip()
)

# ✅ Normalize amount fields (Debit, Credit, Balance)
for col in ["Debit", "Credit", "Balance"]:
    df[col] = (
        df[col]
        .fillna("0")
        .astype(str)
        .str.replace(",", "", regex=False)  # Remove commas
        .str.replace("-", "0", regex=False) # Replace "-" with 0
        .str.strip()
        .replace("", "0")
        .astype(float)
    )

# ✅ Create Transaction Type column
df["Transaction Type"] = df.apply(
    lambda x: "DEBIT" if x["Debit"] > 0 else ("CREDIT" if x["Credit"] > 0 else ""),
    axis=1
)


In [7]:

# Ensure numeric columns
for col in ["Debit", "Credit", "Balance"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

# ✅ Opening Balance is the Balance of first row
opening_balance = df.loc[0, "Balance"]

# ✅ Closing Balance is last row
closing_balance = df["Balance"].iloc[-1]

# ✅ Totals (excluding first row, because first row is already BALANCE AFTER transaction)
total_debit = df["Debit"].iloc[1:].sum()
total_credit = df["Credit"].iloc[1:].sum()

# ✅ Expected Closing Balance
calculated_closing = opening_balance + total_credit - total_debit
diff = closing_balance - calculated_closing

print("📘 BALANCE VALIDATION EQUATION\n")
print(f"Opening Balance : {opening_balance:,.2f}")
print(f"+ Total Credit  : {total_credit:,.2f}")
print(f"- Total Debit   : {total_debit:,.2f}")
print("------------------------------------------------")
print(f"= Calculated Closing Balance : {calculated_closing:,.2f}")
print(f"Actual Closing Balance       : {closing_balance:,.2f}")
print("------------------------------------------------\n")

if abs(diff) < 0.01:
    print("✅ Equation verified! Balances match perfectly.")
else:
    print(f"⚠️ Difference detected: {diff:,.2f}")




📘 BALANCE VALIDATION EQUATION

Opening Balance : 52,236.39
+ Total Credit  : 81,855.00
- Total Debit   : 129,259.99
------------------------------------------------
= Calculated Closing Balance : 4,831.40
Actual Closing Balance       : 4,831.40
------------------------------------------------

✅ Equation verified! Balances match perfectly.


In [8]:
# ✅ Save to Excel & CSV
excel_output = "SBI_Final_Statement.xlsx"
csv_output = "SBI_Final_Statement.csv"

# df.to_excel(excel_output, index=False)
# df.to_csv(csv_output, index=False)

print("\n📁 Files Saved:")
print(f"✅ Excel: {excel_output}")
print(f"✅ CSV  : {csv_output}")


📁 Files Saved:
✅ Excel: SBI_Final_Statement.xlsx
✅ CSV  : SBI_Final_Statement.csv


In [9]:
df

,Date,Details,Ref No./Cheque No,Debit,Credit,Balance,Transaction Type
0,01 DEC 2024,TRANSFER TO 4897690162095 - UPI/DR/43365799200...,,500.0,0.0,52236.39,DEBIT
1,01 DEC 2024,TRANSFER TO 4897690162095 - UPI/DR/47023082961...,,750.0,0.0,51486.39,DEBIT
2,01 DEC 2024,TRANSFER TO 4897690162095 - UPI/DR/43369024733...,,4500.0,0.0,46986.39,DEBIT
3,02 DEC 2024,- ACHDr YESB00709000028661 TP ACH INDIANE,,3000.0,0.0,43986.39,DEBIT
4,02 DEC 2024,TRANSFER TO 4897691162095 - UPI/DR/43371701166...,,500.0,0.0,43486.39,DEBIT
...,...,...,...,...,...,...,...
144,29 DEC 2024,TRANSFER TO 4897690162095 - UPI/DR/47307696586...,,800.0,0.0,5701.40,DEBIT
145,29 DEC 2024,TRANSFER TO 4897690162095 - UPI/DR/43645505184...,,300.0,0.0,5401.40,DEBIT
146,29 DEC 2024,TRANSFER TO 4897690162095 - UPI/DR/43645512698...,,350.0,0.0,5051.40,DEBIT
147,29 DEC 2024,TRANSFER TO 4897690162095 - UPI/DR/43645540343...,,200.0,0.0,4851.40,DEBIT
